In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/311_2026-09-01_to_2026-09-07.csv", dtype=str)
df.shape

(71830, 44)

In [2]:

pd.set_option("display.max_columns", None)
df.head(2)

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,incident_address,street_name,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,facility_type,status,due_date,resolution_description,resolution_action_updated_date,community_board,council_district,police_precinct,bbl,borough,x_coordinate_state_plane,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,70255923,2026-09-01T00:00:05.000,2026-09-01T00:28:25.000,NYPD,New York City Police Department,Noise - Residential,Banging/Pounding,NaN,Residential Building/House,10027,60 WEST 125 STREET,WEST 125 STREET,5 AVENUE,LENOX AVENUE,5 AVENUE,LENOX AVENUE,ADDRESS,NEW YORK,WEST 125 STREET,NaN,Closed,NaN,The New York City Police Department responded ...,2026-09-01T00:28:30.000,10 MANHATTAN,09,Precinct 28,1017227502,MANHATTAN,999650,233401,ONLINE,Unspecified,MANHATTAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.80729427092877,-73.94437110611076,POINT (-73.944371106111 40.807294270929)
1,70266998,2026-09-01T00:00:13.000,2026-09-21T13:13:22.000,HPD,Department of Housing Preservation and Develop...,ELECTRIC,NO LIGHTING,ONE ROOM OR AREA,RESIDENTIAL BUILDING,11230,1362 OCEAN AVENUE,OCEAN AVENUE,NaN,NaN,NaN,NaN,ADDRESS,BROOKLYN,NaN,NaN,Closed,NaN,HPD conducted an inspection of this complaint....,2026-09-21T13:13:22.000,14 BROOKLYN,45,Precinct 70,3067030073,BROOKLYN,996184,168404,ONLINE,Unspecified,BROOKLYN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.62889857195088,-73.95700645258574,POINT (-73.957006452586 40.628898571951)


In [3]:

# request with no id 
df["unique_key"].isna().sum()
#request with repeated id 
df["unique_key"].duplicated().sum()
#fully identical rows
df.duplicated().sum()

#print out
print("missing IDs:", df["unique_key"].isna().sum())
print("repeated IDs:", df["unique_key"].duplicated().sum())
print("fully identical rows:", df.duplicated().sum())




missing IDs: 0
repeated IDs: 0
fully identical rows: 0


In [4]:
#convert text into real date
created = pd.to_datetime(df["created_date"], errors="coerce")

closed = pd.to_datetime(df["closed_date"],errors="coerce")



#print out 
print("created :missing beofre:", df["created_date"].isna().sum())
print("created :missing after:", created.isna().sum())

print("closed :missing before :", df["closed_date"].isna().sum())
print("closed: missing after:" , closed.isna().sum())

#check
print("created range:", created.min(), "to", created.max())
# .min() / .max() -> earliest and latest created date

print("closed range:", closed.min(), "to", closed.max())
# closed dates can run past Sept 7, because a request made

print("closed before created:", (closed < created).sum())


print("still open (no closed_date):", closed.isna().sum())


created :missing beofre: 0
created :missing after: 0
closed :missing before : 5968
closed: missing after: 5968
created range: 2026-09-01 00:00:05 to 2026-09-07 23:59:49
closed range: 2026-08-31 08:57:00 to 2026-09-29 21:25:09
closed before created: 25
still open (no closed_date): 5968


In [5]:
#Find the bad rows
bad = df[closed<created]
bad[["unique_key", "agency", "complaint_type", "created_date", "closed_date", "status"]]

print(bad["agency"].value_counts())
print(df["agency"].value_counts().head(5))

gap = (created-closed)[closed<created]
print("the gap:",gap.describe())

agency
DEP    23
DOT     2
Name: count, dtype: int64
agency
NYPD    36474
HPD     10995
DSNY     6443
DOT      4485
DEP      3772
Name: count, dtype: int64
the gap: count                        25
mean     0 days 02:17:42.360000
std      0 days 09:42:11.328040
min             0 days 00:00:02
25%             0 days 00:00:12
50%             0 days 00:00:18
75%             0 days 00:00:23
max             2 days 00:01:00
dtype: object


In [6]:
#Check 25 bad rows
bad[["unique_key", "agency", "complaint_type", "created_date", "closed_date", "status"]]

print("DEP share of all requests (%):", round((df["agency"] == "DEP").mean() * 100, 1))

print("DEP share of bad rows (%):", round((bad["agency"] == "DEP").mean() * 100, 1))

print("DEP requests with this problem (%):", round(23 / (df["agency"] == "DEP").sum() * 100, 2))

print("gaps under 1 minute:", (gap < pd.Timedelta("1min")).sum(), "of", len(gap))

print(bad["status"].value_counts())


DEP share of all requests (%): 5.3
DEP share of bad rows (%): 92.0
DEP requests with this problem (%): 0.61
gaps under 1 minute: 23 of 25
status
Closed     24
Pending     1
Name: count, dtype: int64


In [7]:
#largest gap row
gap.sort_values(ascending=False).head(3)


11830   2 days 00:01:00
15926   0 days 09:15:29
6092    0 days 00:00:30
dtype: timedelta64[us]

In [8]:
#A new table show "bad"
bad_view = bad[["unique_key", "agency", "complaint_type", "created_date", "closed_date", "status"]].copy()

#add gap
bad_view["gap"] = gap
#sorting the gap 
bad_view.sort_values("gap", ascending=False)


,unique_key,agency,complaint_type,created_date,closed_date,status,gap
11830,70273435,DOT,Street Light Condition,2026-09-02T08:58:00.000,2026-08-31T08:57:00.000,Pending,2 days 00:01:00
15926,70290962,DOT,Street Condition,2026-09-02T15:15:29.000,2026-09-02T06:00:00.000,Closed,0 days 09:15:29
6092,70268983,DEP,Water Maintenance,2026-09-01T15:24:30.000,2026-09-01T15:24:00.000,Closed,0 days 00:00:30
56774,70320856,DEP,Water Maintenance,2026-09-06T16:46:26.000,2026-09-06T16:46:00.000,Closed,0 days 00:00:26
46935,70306898,DEP,Hazardous Material,2026-09-05T17:00:26.000,2026-09-05T17:00:00.000,Closed,0 days 00:00:26
56347,70319505,DEP,Water Maintenance,2026-09-06T15:45:25.000,2026-09-06T15:45:00.000,Closed,0 days 00:00:25
15541,70281168,DEP,Water Maintenance,2026-09-02T14:36:23.000,2026-09-02T14:36:00.000,Closed,0 days 00:00:23
46742,70308156,DEP,Water Maintenance,2026-09-05T16:30:22.000,2026-09-05T16:30:00.000,Closed,0 days 00:00:22
23699,70291153,DEP,Sewer Maintenance,2026-09-03T11:22:20.000,2026-09-03T11:22:00.000,Closed,0 days 00:00:20
23856,70294096,DEP,Water Maintenance,2026-09-03T11:36:20.000,2026-09-03T11:36:00.000,Closed,0 days 00:00:20


In [9]:
is_tiny = gap < pd.Timedelta("1min")

#new table only bad["agency"] , is_tiny
pd.crosstab(bad["agency"], is_tiny)


col_0,False,True
agency,,
DEP,0,23
DOT,2,0


In [10]:
#Pending
bad_view[bad_view["status"] != "Closed"]


,unique_key,agency,complaint_type,created_date,closed_date,status,gap
11830,70273435,DOT,Street Light Condition,2026-09-02T08:58:00.000,2026-08-31T08:57:00.000,Pending,2 days 00:01:00


In [11]:
dep_bad = bad[bad["agency"] == "DEP"]
# the 23 bad DEP rows

same_minute = closed[dep_bad.index] == created[dep_bad.index].dt.floor("min")
# created[...].dt.floor("min") -> rounds each created time DOWN to the minute


print("DEP bad rows where closed = created rounded down:", same_minute.sum(), "of", len(dep_bad))
# if the answer is 23 of 23, the pattern is exact

DEP bad rows where closed = created rounded down: 23 of 23


In [12]:
dep_closed = closed[df["agency"] == "DEP"].dropna()
# closed times of DEP requests only

other_closed = closed[df["agency"] != "DEP"].dropna()
# the same for every other agency

print("DEP closed times with 0 seconds (%):", round((dep_closed.dt.second == 0).mean() * 100, 1))


print("Other agencies, 0 seconds (%):", round((other_closed.dt.second == 0).mean() * 100, 1))


DEP closed times with 0 seconds (%): 91.8
Other agencies, 0 seconds (%): 8.2


In [13]:
tmp = pd.DataFrame({"agency": df["agency"], "zero_sec": closed.dt.second == 0})


tmp = tmp[closed.notna()]
# keep only requests that actually have a close time

print(tmp.groupby("agency")["zero_sec"].agg(["mean", "count"]).sort_values("mean", ascending=False))



            mean  count
agency                 
DEP     0.918147   3604
DOB     0.653879   2346
DOT     0.496310   3794
DSNY    0.139862   6249
OOS     0.038462     26
DCWP    0.032432    185
DOHMH   0.023233   1033
DPR     0.018483   1569
NYPD    0.016615  36474
HPD     0.012460   9470
DHS     0.009202    978
OTI     0.000000      2
TLC     0.000000    132


In [14]:
print(df["status"].value_counts(dropna=False))
#count all value even empty value 

has_closed_date = df["closed_date"].notna()
pd.crosstab(df["status"], has_closed_date.rename("has_closed_date"), margins=True)

status
Closed         64898
In Progress     4127
Open            2535
Assigned         229
Pending           34
Unspecified        7
Name: count, dtype: int64


has_closed_date,False,True,All
status,,,
Assigned,83,146,229
Closed,0,64898,64898
In Progress,4013,114,4127
Open,1839,696,2535
Pending,33,1,34
Unspecified,0,7,7
All,5968,65862,71830


In [15]:
# Isoalating 964 requests have a close date but aren't marked Closed
 
status_conflict = has_closed_date & (df["status"] != "Closed")

print("close date but status not Closed:", status_conflict.sum())


close date but status not Closed: 964


In [16]:
#Find which agencies
print(pd.crosstab(df["agency"], status_conflict.rename("status_conflict")))


status_conflict  False  True 
agency                       
DCWP               430      0
DEP               3653    119
DHS               1018      0
DOB               1536    810
DOHMH             1780      0
DOT               4456     29
DPR               3215      1
DSNY              6443      0
EDC                 89      0
HPD              10991      4
NYPD             36473      1
OOS                106      0
OTI                  2      0
TLC                674      0


In [17]:
#Which complaint type
print(df[status_conflict]["complaint_type"].value_counts().head(10))


complaint_type
Building/Use                               363
General Construction/Plumbing              202
Elevator                                   154
Water Maintenance                           52
Special Projects Inspection Team (SPIT)     46
Plumbing                                    32
Noise                                       31
Sewer Maintenance                           29
Traffic Signal Condition                    28
Water Quality                                6
Name: count, dtype: int64


In [18]:
dob = df[df["agency"] == "DOB"]
# keep only the DOB requests

print("DOB requests:", len(dob))
print("DOB with no closed_date:", dob["closed_date"].isna().sum())

print(dob["status"].value_counts(dropna=False))
# the status mix for DOB

DOB requests: 2346
DOB with no closed_date: 0
status
Closed      1536
Open         692
Assigned     118
Name: count, dtype: int64


In [19]:
dob_time = (closed - created)[df["agency"] == "DOB"]

dob_status = df.loc[df["agency"] == "DOB", "status"]

print(dob_time.groupby(dob_status).median())


status
Assigned   12 days 12:32:26.500000
Closed      2 days 00:28:43.500000
Open               0 days 00:00:00
dtype: timedelta64[us]


In [20]:
# numbers DOB Open rows have closed = created
dob_open = df[(df["agency"] == "DOB") & (df["status"] == "Open")]
zero_gap = closed[dob_open.index] == created[dob_open.index]

print("DOB Open with closed == created:", zero_gap.sum(), "of", len(dob_open))
# .sum() counts the Trues

DOB Open with closed == created: 692 of 692


In [21]:
#checking if DOB the only one
exact = (closed == created)[status_conflict]

print(exact.groupby([df.loc[status_conflict, "agency"], df.loc[status_conflict, "status"]]).agg(["sum", "count"]))


                    sum  count
agency status                 
DEP    In Progress    0    114
       Unspecified    0      5
DOB    Assigned       3    118
       Open         692    692
DOT    Assigned       0     28
       Pending        0      1
DPR    Unspecified    0      1
HPD    Open           0      4
NYPD   Unspecified    0      1


In [22]:
#cheking if whether DOBs closed date = another date column

updated = pd.to_datetime(df["resolution_action_updated_date"], errors="coerce")
# convert the "last updated" column to real dates (unreadable values become NaT)

dob_mask = df["agency"] == "DOB"
# True for DOB rows, False for the rest

print("DOB: closed == resolution_action_updated_date (%):",round((closed[dob_mask] == updated[dob_mask]).mean() * 100, 1))
# compare the two date columns row by row for DOB, then take the share that match

print("DOB: resolution_action_updated_date missing:", updated[dob_mask].isna().sum())


DOB: closed == resolution_action_updated_date (%): 64.7
DOB: resolution_action_updated_date missing: 0


In [23]:
#checking the all 44 collums
profile = pd.DataFrame({
    "missing_pct": (df.isna().mean() * 100).round(1),
    # for each column: percent of values that are empty
    "distinct": df.nunique(),
    # for each column: how many different values it has
})
# pd.DataFrame builds a table from the two columns above, one row per original column

print(profile.sort_values("missing_pct", ascending=False).to_string())


                                missing_pct  distinct
taxi_company_borough                   99.9         5
due_date                               99.7       169
facility_type                          99.7         1
road_ramp                              99.7        35
bridge_highway_direction               99.7        57
bridge_highway_name                    99.5        47
bridge_highway_segment                 99.5       121
taxi_pick_up_location                  99.1       449
vehicle_type                           94.9         8
descriptor_2                           71.4       471
landmark                               29.7      4434
cross_street_1                         21.7      5043
cross_street_2                         21.6      5048
intersection_street_1                  21.6      4661
intersection_street_2                  21.5      4729
bbl                                    11.9     32474
closed_date                             8.3     41038
location_type               

In [24]:
#which DOB rows = updated
match_updated = (closed == updated)[dob_mask]

print(match_updated.groupby(df.loc[dob_mask, "status"]).agg(["sum", "count"]))

           sum  count
status               
Assigned   115    118
Closed    1402   1536
Open         0    692


In [25]:
# facility_types single value
print(df["facility_type"].value_counts(dropna=False))


facility_type
NaN            71634
DSNY Garage      196
Name: count, dtype: int64


In [26]:
#see all 44 columns of the profile
pd.set_option("display.max_rows", None)
# tell pandas never to hide rows when it shows a table

profile.sort_values("missing_pct", ascending=False)

,missing_pct,distinct
taxi_company_borough,99.9,5
due_date,99.7,169
facility_type,99.7,1
road_ramp,99.7,35
bridge_highway_direction,99.7,57
bridge_highway_name,99.5,47
bridge_highway_segment,99.5,121
taxi_pick_up_location,99.1,449
vehicle_type,94.9,8
descriptor_2,71.4,471


In [27]:
#checking borough

print(df["borough"].value_counts(dropna=False))
# every distinct borough value with its count, includes empty ones

print(df["park_borough"].value_counts(dropna=False))
# the same check for park_borough, which also has 6 distinct values

borough
BROOKLYN         23495
QUEENS           17933
MANHATTAN        14055
BRONX            13459
STATEN ISLAND     2808
Unspecified         80
Name: count, dtype: int64
park_borough
BROOKLYN         23495
QUEENS           17933
MANHATTAN        14055
BRONX            13459
STATEN ISLAND     2808
Unspecified         80
Name: count, dtype: int64


In [28]:
#checking agency_name just a longer name for agency?

pairs = df[["agency", "agency_name"]].drop_duplicates()
# .drop_duplicates()            -> keep each distinct (agency, agency_name) combination once

print("distinct pairs:", len(pairs), "| agency:", df["agency"].nunique(), "| agency_name:", df["agency_name"].nunique())
# if all three numbers are 14, each agency code maps to exactly one name

distinct pairs: 14 | agency: 14 | agency_name: 14


In [29]:
#checking are the location columns empty on the same rows?

print(pd.crosstab(df["latitude"].isna().rename("lat_missing"), df["location"].isna().rename("location_missing")))


location_missing  False  True 
lat_missing                   
False             70529      0
True                  0   1301


In [30]:
#are the nearly-empty columns empty because they don't apply to most requests, or because values got lost?
for col in ["taxi_company_borough", "vehicle_type", "road_ramp", "bridge_highway_name", "facility_type"]:
    types = df.loc[df[col].notna(), "complaint_type"].value_counts().head(3)
    # df[col].notna()                -> True where this column HAS a value
    # df.loc[..., "complaint_type"]  -> the complaint types of those rows only
    # .value_counts().head(3)        -> the 3 most common types among them
    print(col, "->", dict(types))

taxi_company_borough -> {'For Hire Vehicle Complaint': np.int64(56), 'For Hire Vehicle Report': np.int64(2)}
vehicle_type -> {'Abandoned Vehicle': np.int64(1441), 'Noise - Vehicle': np.int64(1247), 'Illegal Parking': np.int64(941)}
road_ramp -> {'Highway Condition': np.int64(119), 'Encampment': np.int64(27), 'Homeless Person Assistance': np.int64(27)}
bridge_highway_name -> {'Highway Condition': np.int64(119), 'Panhandling': np.int64(79), 'Homeless Person Assistance': np.int64(76)}
facility_type -> {'Derelict Vehicles': np.int64(196)}
